# Praktikum 3 - Exact NN dan HNSW

**Nama:** Agus

**NIM:** 244107020049

[Sumber modul](https://polinema.gitbook.io/jti-modul-praktikum-pembelajaran-mesin-2025-2026/js06-ann-approximate-nearest-neighbors/praktikum-3)

## Langkah 0 - Persiapan Lingkungan

In [1]:
import sys
import numpy as np
np.random.seed(42)
print("Kernel:", sys.executable)

import faiss
faiss.omp_set_num_threads(4)


Kernel: /Volumes/External/miniconda3/envs/ml-praktikum/bin/python


## Langkah 1 - Import Library

In [2]:
import hnswlib
import numpy as np
import time
from sklearn.neighbors import NearestNeighbors

# ===========================


## Langkah 2 - Buat data 2D acak

In [3]:
# 1. Buat data 2D acak
# ===========================
num_elements = 1000
dim = 2
data = np.random.random((num_elements, dim)).astype(np.float32)

# Query point
query = np.array([[0.5, 0.5]], dtype=np.float32)
k = 5  # cari 5 tetangga terdekat

# ===========================


## Langkah 3 - Exact NN (Brute Force)

In [4]:
# 2. Exact NN (Brute Force)
# ===========================
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(data)

start = time.time()
distances, indices = nn.kneighbors(query)
end = time.time()

print("=== Exact NN ===")
print("Indices:", indices)
print("Distances:", distances)
print("Waktu:", end - start, "detik")

# ===========================


=== Exact NN ===
Indices: [[137 170 750 316 991]]
Distances: [[0.01144331 0.0278216  0.02824018 0.03417073 0.04514859]]
Waktu: 0.08278393745422363 detik


## Langkah 4 - HNSW

In [5]:
# 3. HNSW
# ===========================
# Inisialisasi index HNSW
p = hnswlib.Index(space='l2', dim=dim)  

# Ukuran maksimum elemen yang bisa ditampung
p.init_index(max_elements=num_elements, ef_construction=100, M=16)

# Tambahkan data
p.add_items(data)

# Set parameter pencarian
p.set_ef(50)   # tradeoff speed vs accuracy

start = time.time()
labels, distances = p.knn_query(query, k=k)
end = time.time()

print("\n=== HNSW ===")
print("Indices:", labels)
print("Distances:", distances)
print("Waktu:", end - start, "detik")



=== HNSW ===
Indices: [[137 170 750 316 991]]
Distances: [[0.00013095 0.00077404 0.00079751 0.00116764 0.0020384 ]]
Waktu: 3.504753112792969e-05 detik


## Percobaan Tambahan - Variasi Metrik, Ukuran, dan Dimensi

Bandingkan 1.000 dan 1.000.000 baris, 2D dan 5D, serta Euclidean dan Cosine. Setiap pasangan memakai dataset/query dan k yang sama. Exact menggunakan FAISS Flat sebagai referensi. Cosine menggunakan normalisasi L2 pada FAISS. Satu query memberi pengukuran ilustratif, bukan benchmark latency yang stabil.

FAISS dan HNSW l2 mengembalikan **jarak Euclidean kuadrat**; scikit-learn NearestNeighbors mengembalikan jarak Euclidean. Urutan tetangga tetap dapat dibandingkan, tetapi nilai jaraknya harus disetarakan lebih dahulu.

In [6]:
import gc
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from annoy import AnnoyIndex
import faiss
import hnswlib


In [7]:
def recall_at_k(indices, truth, k):
    return sum(len(set(a[:k]) & set(b[:k])) for a, b in zip(indices, truth)) / (len(truth) * k)

def prepare_metric(X, query, metric):
    X = np.ascontiguousarray(X, dtype=np.float32)
    query = np.ascontiguousarray(query, dtype=np.float32)
    if metric == 'cosine':
        X = X.copy()
        query = query.copy()
        faiss.normalize_L2(X)
        faiss.normalize_L2(query)
    return X, query

def experiment_ann(n, dim, metric, algorithm, trees=8):
    rng = np.random.RandomState(42)
    X = rng.random_sample((n, dim)).astype(np.float32)
    query = rng.random_sample((1, dim)).astype(np.float32)
    X, query = prepare_metric(X, query, metric)
    k = 5
    start = time.perf_counter()
    exact = faiss.IndexFlatL2(dim) if metric == 'euclidean' else faiss.IndexFlatIP(dim)
    exact.add(X)
    exact_build = time.perf_counter() - start
    start = time.perf_counter()
    exact_D, truth = exact.search(query, k)
    exact_time = time.perf_counter() - start
    start = time.perf_counter()
    if algorithm == 'Annoy':
        index = AnnoyIndex(dim, 'euclidean' if metric == 'euclidean' else 'angular')
        index.set_seed(42)
        for i, vec in enumerate(X):
            index.add_item(i, vec)
        index.build(trees)
        build = time.perf_counter() - start
        start = time.perf_counter()
        ids, values = index.get_nns_by_vector(query[0], k, include_distances=True)
        pred = np.array([ids])
    elif algorithm == 'FAISS IVF+PQ':
        native = faiss.METRIC_L2 if metric == 'euclidean' else faiss.METRIC_INNER_PRODUCT
        quantizer = faiss.IndexFlatL2(dim) if metric == 'euclidean' else faiss.IndexFlatIP(dim)
        # m harus membagi dim: m=2 untuk 2D, m=5 untuk 5D.
        index = faiss.IndexIVFPQ(quantizer, dim, 10, dim, 8, native)
        index.train(X)
        index.add(X)
        build = time.perf_counter() - start
        start = time.perf_counter()
        values, pred = index.search(query, k)
    else:
        index = hnswlib.Index(space='l2' if metric == 'euclidean' else 'cosine', dim=dim)
        index.init_index(max_elements=n, ef_construction=100, M=16, random_seed=42)
        index.add_items(X)
        index.set_ef(50)
        build = time.perf_counter() - start
        start = time.perf_counter()
        pred, values = index.knn_query(query, k=k)
    query_time = time.perf_counter() - start
    result = {'Model': algorithm, 'Metrik': metric, 'Dimensi': dim, 'Jumlah data': n,
        'Tree': trees if algorithm == 'Annoy' else np.nan,
        'Index Exact': truth[0].tolist(), 'Index ANN': pred[0].tolist(),
        'Recall@5': recall_at_k(pred, truth, k),
        'Build Exact (s)': exact_build, 'Build ANN (s)': build,
        'Query Exact (ms)': exact_time * 1000, 'Query ANN (ms)': query_time * 1000}
    del exact, index, X
    gc.collect()
    return result


In [8]:
experiment_rows = []
for n in [1000, 1_000_000]:
    for dim in [2, 5]:
        for metric in ['euclidean', 'cosine']:
            print(f'Eksperimen: n={n}, dim={dim}, metric={metric}')
            experiment_rows.append(experiment_ann(n, dim, metric, 'HNSW'))
display(pd.DataFrame(experiment_rows).round(6))


Eksperimen: n=1000, dim=2, metric=euclidean


Eksperimen: n=1000, dim=2, metric=cosine


Eksperimen: n=1000, dim=5, metric=euclidean
Eksperimen: n=1000, dim=5, metric=cosine
Eksperimen: n=1000000, dim=2, metric=euclidean


Eksperimen: n=1000000, dim=2, metric=cosine


Eksperimen: n=1000000, dim=5, metric=euclidean


Eksperimen: n=1000000, dim=5, metric=cosine


,Model,Metrik,Dimensi,Jumlah data,Tree,Index Exact,Index ANN,Recall@5,Build Exact (s),Build ANN (s),Query Exact (ms),Query ANN (ms)
0,HNSW,euclidean,2,1000,NaN,"[112, 535, 777, 246, 763]","[112, 535, 777, 246, 763]",1.0,0.000166,0.003234,0.027750,0.011958
1,HNSW,cosine,2,1000,NaN,"[738, 183, 561, 91, 244]","[738, 183, 561, 91, 244]",1.0,0.000018,0.003020,0.068750,0.011083
2,HNSW,euclidean,5,1000,NaN,"[988, 780, 27, 943, 93]","[988, 780, 27, 943, 93]",1.0,0.000015,0.004034,0.011084,0.016708
3,HNSW,cosine,5,1000,NaN,"[988, 943, 780, 332, 27]","[988, 943, 780, 332, 27]",1.0,0.000012,0.003753,0.011792,0.014917
4,HNSW,euclidean,2,1000000,NaN,"[132774, 119034, 511191, 482199, 913140]","[132774, 119034, 511191, 482199, 913140]",1.0,0.000687,3.626764,0.647709,0.025709
5,HNSW,cosine,2,1000000,NaN,"[737055, 47587, 43272, 18017, 8795]","[878208, 900745, 918997, 919879, 960401]",0.0,0.000649,6.029918,0.204833,0.031500
6,HNSW,euclidean,5,1000000,NaN,"[901095, 495168, 561017, 163986, 185734]","[901095, 495168, 561017, 163986, 185734]",1.0,0.001916,5.543882,2.309542,0.034208
7,HNSW,cosine,5,1000000,NaN,"[653774, 135588, 561017, 901095, 211473]","[653774, 135588, 561017, 901095, 211473]",1.0,0.001288,5.678321,1.987042,0.114000


## Analisis

Recall@5 menunjukkan bagian tetangga exact yang ditemukan. Dimensi dan metrik mengubah urutan kedekatan. Waktu build dipisahkan dari waktu query. Hasil satu query perlu dibaca bersama recall, bukan waktu saja.

## Satuan jarak

HNSW `space="l2"` mengembalikan jarak Euclidean kuadrat; `NearestNeighbors(metric="euclidean")` mengembalikan jarak Euclidean. Perbandingan ID tetangga dan recall tetap valid. Nilai jarak native kedua keluaran tidak dibandingkan secara langsung.